# 06 — SFMS: fixed SDSS relation vs the median TNG100 / TNG50 sequence (z = 0)

Compares the **star-forming main sequence** used to define "quenched" in the pipeline against the
**actual median SFMS measured from TNG**, at z = 0, for TNG100 and TNG50 — using the already-run
catalog from `01_generate_catalogs_massive.ipynb`
(`tng_satellites_hostlogM12.0plus_logM7.00.csv`).

* **Fixed SDSS relation** (what the pipeline currently uses):
  $\log_{10}\mathrm{SFR}_{\rm MS} = 0.75\,\log_{10}M_* - 7.5$ (Martín-Navarro+ 2021), with
  quenched $\equiv$ 1 dex below it.
* **Median TNG SFMS**: the running median of $\log_{10}\mathrm{SFR}$ vs $\log_{10}M_*$ for the
  star-forming ($\mathrm{SFR}>0$) satellites in the catalog, per simulation.

The point is to see the **offset** between the fixed threshold and the true TNG sequence — a
~0.2–0.3 dex offset is enough to change which satellites count as quenched (and can flip a small
quench-anisotropy amplitude $b$).

> **Caveat.** This catalog holds only **satellites of $\log M_{200c}>12$ hosts**, not the full
> galaxy population. Satellites are more quenched than centrals, so this median sits somewhat
> *below* the true (all-galaxy) TNG SFMS the A&A 2025 paper fits. For an unbiased SFMS you would
> fit from every galaxy at z=0 in `notebooks2/01` — this notebook is the quick look from data you
> already have. Masses are $\log_{10}$ physical $M_\odot$; SFR is $M_\odot\,\mathrm{yr}^{-1}$.

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
%matplotlib inline
mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "axes.unicode_minus": False,
})

# ================= CONFIG =================
DATA_ROOT = "../data2"
BASE_CAT  = "tng_satellites_hostlogM12.0plus_logM7.00.csv"   # from 01_generate_catalogs_massive
REDZ      = "z0"                                             # z = 0

SIMS = [("tng100", "#1f77b4"), ("tng50", "#d62728")]

# mass bins for the running median (log10 M*/Msun)
MASS_EDGES = np.arange(7.0, 11.51, 0.25)
MIN_PER_BIN = 10                     # require this many SF galaxies to plot a median point
# =========================================

def sdss_sfms(logm):
    '''Fixed SDSS SFMS used by the pipeline: log10 SFR_MS = 0.75 log10 M* - 7.5.'''
    return 0.75 * np.asarray(logm) - 7.5

MASS_CENT = 0.5 * (MASS_EDGES[:-1] + MASS_EDGES[1:])
print(f"comparing fixed SDSS SFMS vs median TNG SFMS at {REDZ} for {[s for s,_ in SIMS]}")

## Load the catalogs and compute the median TNG SFMS

For each simulation we take the star-forming satellites ($\mathrm{SFR}>0$) and compute, per stellar
mass bin, the median $\log_{10}\mathrm{SFR}$ and the 16–84th percentile spread. We also record the
fraction with $\mathrm{SFR}=0$ (the TNG resolution floor) per bin — those are excluded from the
median but count as quenched.

In [ ]:
def median_sfms(df):
    '''Running median log10(SFR) vs log10(M*) for SFR>0; also the SFR=0 fraction per bin.'''
    logm = df["mstar_phys"].to_numpy()
    sfr  = df["sfr"].to_numpy()
    sf   = sfr > 0
    logsfr = np.full(len(sfr), np.nan)
    logsfr[sf] = np.log10(sfr[sf])
    med = np.full(len(MASS_CENT), np.nan)
    lo  = np.full(len(MASS_CENT), np.nan)
    hi  = np.full(len(MASS_CENT), np.nan)
    f0  = np.full(len(MASS_CENT), np.nan)
    idx = np.digitize(logm, MASS_EDGES) - 1
    for j in range(len(MASS_CENT)):
        inbin = idx == j
        if inbin.sum() == 0:
            continue
        f0[j] = np.mean(~sf[inbin])                       # SFR=0 fraction in this mass bin
        sfbin = inbin & sf
        if sfbin.sum() >= MIN_PER_BIN:
            vals = logsfr[sfbin]
            med[j], lo[j], hi[j] = np.percentile(vals, [50, 16, 84])
    return dict(logm=logm, logsfr=logsfr, sf=sf, med=med, lo=lo, hi=hi, f0=f0)

sfms = {}
for sim, _ in SIMS:
    path = os.path.join(DATA_ROOT, sim, REDZ, BASE_CAT)
    if not os.path.exists(path):
        print(f"[skip] missing {path}")
        continue
    df = pd.read_csv(path)
    sfms[sim] = median_sfms(df)
    n_sf = int(sfms[sim]["sf"].sum())
    print(f"{sim:6s}: {len(df):6d} satellites  ({n_sf} with SFR>0, {len(df)-n_sf} at SFR=0)")

## Offset table — median TNG SFMS minus the fixed SDSS relation

Positive = TNG sits *above* the SDSS line; negative = TNG sits *below* it (so the fixed threshold
is too high and over-counts quenched satellites at that mass).

In [ ]:
print(f"{'logM*':>7s} | " + " | ".join(f"{s+' med':>9s}  {'-SDSS':>7s}" for s, _ in SIMS if s in sfms))
for j, mc in enumerate(MASS_CENT):
    row = f"{mc:7.2f} | "
    cols = []
    for sim, _ in SIMS:
        if sim not in sfms:
            continue
        m = sfms[sim]["med"][j]
        if np.isfinite(m):
            cols.append(f"{m:9.2f}  {m - sdss_sfms(mc):+7.2f}")
        else:
            cols.append(f"{'--':>9s}  {'--':>7s}")
    print(row + " | ".join(cols))

## Figure — SDSS SFMS (dashed) vs median TNG SFMS (solid), per simulation

Faint points = star-forming satellites ($\mathrm{SFR}>0$); solid line + band = median TNG SFMS
(16–84%); black dashed = fixed SDSS SFMS; black dotted = the SDSS $-1$ dex quench threshold the
pipeline currently uses. Where the TNG median sits below the dotted line, the fixed cut misclassifies.

In [ ]:
xline = np.linspace(MASS_EDGES[0], MASS_EDGES[-1], 100)
fig, axes = plt.subplots(1, len(SIMS), figsize=(6.5 * len(SIMS), 5.2), sharex=True, sharey=True)
if len(SIMS) == 1:
    axes = [axes]
for ax, (sim, color) in zip(axes, SIMS):
    if sim not in sfms:
        ax.set_visible(False); continue
    s = sfms[sim]
    ax.scatter(s["logm"][s["sf"]], s["logsfr"][s["sf"]], s=3, color=color, alpha=0.12, lw=0)
    ok = np.isfinite(s["med"])
    ax.plot(MASS_CENT[ok], s["med"][ok], color=color, lw=2.5, label=f"{sim.upper()} median SFMS")
    ax.fill_between(MASS_CENT[ok], s["lo"][ok], s["hi"][ok], color=color, alpha=0.15)
    ax.plot(xline, sdss_sfms(xline), color="k", ls="--", lw=2, label="SDSS SFMS (0.75 logM* - 7.5)")
    ax.plot(xline, sdss_sfms(xline) - 1.0, color="k", ls=":", lw=1.5, label="SDSS SFMS - 1 dex (quench cut)")
    ax.set_xlim(MASS_EDGES[0], MASS_EDGES[-1]); ax.set_ylim(-5, 2)
    ax.set_xlabel(r"$\log_{10}(M_*/M_\odot)$")
    ax.set_title(sim.upper())
    ax.legend(fontsize=9, fancybox=False, edgecolor="k", loc="lower right")
    ax.tick_params(which="both", direction="in", top=True, right=True)
axes[0].set_ylabel(r"$\log_{10}(\mathrm{SFR}/M_\odot\,\mathrm{yr}^{-1})$")
fig.suptitle(f"Star-forming main sequence: fixed SDSS vs median TNG  ({REDZ}, satellites of logM200c>12 hosts)",
             y=1.02, fontsize=13)
plt.subplots_adjust(wspace=0.06); plt.show()

## SFR = 0 fraction vs stellar mass

The share of satellites at exactly $\mathrm{SFR}=0$ (the TNG resolution floor) — these are counted
as quenched regardless of the SFMS choice, and dominate the low-mass, quenched population.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for sim, color in SIMS:
    if sim not in sfms:
        continue
    f0 = sfms[sim]["f0"]; ok = np.isfinite(f0)
    ax.plot(MASS_CENT[ok], f0[ok], "o-", color=color, label=sim.upper())
ax.set_xlabel(r"$\log_{10}(M_*/M_\odot)$"); ax.set_ylabel("fraction with SFR = 0")
ax.set_ylim(0, 1); ax.set_title(f"SFR = 0 fraction vs stellar mass ({REDZ})")
ax.legend(fancybox=False, edgecolor="k")
ax.tick_params(which="both", direction="in", top=True, right=True)
plt.tight_layout(); plt.show()